In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/544 Project/Train"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from dataclasses import dataclass
from typing import Any
import torch
from torchinfo import summary
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForSeq2Seq, DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType
import evaluate


In [ ]:
df = pd.read_excel("./data/train/truthfulqa_generation_data.xlsx")
df.head()

In [ ]:
def format_dataset(df):
    examples = []
    for _, row in df.iterrows():
        question = row["Question"]
        examples.append({"question": question, "answer": row["Best Answer"], "label": "correct"})
        for ans in row["Correct Answers"].split(";"):
            ans = ans.strip()
            if ans and ans != row["Best Answer"]:
                examples.append({"question": question, "answer": ans, "label": "correct"})
    return pd.DataFrame(examples)

In [ ]:
formatted_df = format_dataset(df)
formatted_df.head()

In [ ]:
# ADD Hugging Face API KEY instead of HF_TOKEN
login(token="HF_TOKEN")

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"

In [ ]:
def apply_chat_template(row):
    messages = [
        {
            "role": "user",
            "content": (
                "You are a helpful and truthful assistant. "
                "Answer the following question accurately and honestly. "
                "If you are unsure, say so rather than guessing.\n\n"
                f"Question: {row['question']}"
            ),
        },
        {
            "role": "assistant",
            "content": row["answer"],
        },
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [ ]:
hf_dataset = Dataset.from_pandas(formatted_df[["question", "answer"]])
hf_dataset = hf_dataset.map(apply_chat_template)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
model = get_peft_model(model, lora_config)
summary(model)

In [ ]:
@dataclass
class Gemma3TextCollator:
    base_collator: Any
    def __call__(self, features):
        batch = self.base_collator(features)
        batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        return batch

In [ ]:
base_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)
collator = Gemma3TextCollator(base_collator=base_collator)

In [ ]:
training_args = SFTConfig(
    output_dir="./train/gemma",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    max_length=512,
    dataset_text_field="text",
    packing=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=hf_dataset,
    eval_dataset=hf_dataset,
    args=training_args,
    data_collator=collator,
)

In [ ]:
pre_eval = trainer.evaluate()
print(f"Baseline {MODEL_ID} Model: ")
for k, v in pre_eval.items():
    print(f"  {k}: {v}")

In [ ]:
trainer.train()

In [ ]:
post_eval = trainer.evaluate()
print(f"LoRA Fine-tuned {MODEL_ID} Model: ")
for k, v in post_eval.items():
    print(f"  {k}: {v}")

In [ ]:
print("Improvements: ")
for k in post_eval:
    if k in pre_eval and isinstance(post_eval[k], float):
        delta = post_eval[k] - pre_eval[k]
        print(f"  {k}: {'+' if delta > 0 else ''}{delta:.4f}")

In [ ]:
model.save_pretrained("./train/gemma/model_files/model/model")
tokenizer.save_pretrained("./train/gemma/model_files/tokenizer")